In [36]:
import os
import math
import csv
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Dict, Counter


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
# sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# from Common.Utils import save_training_results


In [37]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 300
    n_nodes: int = 3
    n_users: int = 200
    step_size: float = 10.0
    arrival_rate: float = 10.0  # users per second
    zipf_alpha: float = 0.5
    n_videos: int = 500
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.1  # 10% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_b: float = (
        n_gops * n_tiles * base_tile_b +
        n_gops * tiles_per_viewport * enh_tile_b
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    buffer_capacity: int = 10000
    window_len: int = 3  # LSTM sequence length (history window)
    nb_interval: int = 5  # train every 5 requests
    
    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int: # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int: # |A| = 5C + 1 (Section VI-B)
        return (self.cache_size + self.cache_size * self.tiles_per_viewport + 1)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DQN(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        hidden = action_dim  # = 5C + 1
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)  # linear

class ReplayBuffer:
    def __init__(self, capacity: int = 2000):
        self.memory = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))
    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

class DQNAgent:
    def __init__(self, cfg: Config):
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim
        self.epsilon = cfg.epsilon_start
        self.epsilon_min = cfg.epsilon_min
        self.epsilon_decay = cfg.epsilon_decay
        self.gamma = cfg.gamma
        self.batch_size = cfg.batch_size
        self.buffer = ReplayBuffer(cfg.buffer_capacity)
        self.policy_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()
        self.nb_interval = cfg.nb_interval  # train every 5 requests

        self.capacity = cfg.cache_size

    def select_action(self, state, idx: int = 0, g: int = 0):
        if random.random() < self.epsilon:
            if g == 0:
                return random.randint(0, self.capacity), None
            else:
                offset = self.capacity + idx * 4
                action = random.randint(offset + 1, offset + 5)

                return 0 if action == offset + 5 else action, None

        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)

        if g == 0:
            return q_values[0, 0:self.capacity + 1].argmax().item(), q_values
        else:
            offset = self.capacity + idx * 4
            slice_vals = q_values[0, offset + 1 : offset + 5]
            max_slice, max_idx = slice_vals.max(0)
            action = 0 if q_values[0, 0] >= max_slice else (offset + 1 + max_idx.item())

            return action, q_values

    def remember(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)
        s = torch.tensor(np.stack(s), dtype=torch.float32).to(device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(device)
        a = torch.tensor(a, dtype=torch.int64).to(device)
        r = torch.tensor(r, dtype=torch.float32).to(device)
        d = torch.tensor(d, dtype=torch.float32).to(device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_net(ns).max(1)[0]
            target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [39]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': total_reward,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'epsilon': agent.epsilon if agent else None
        })


In [40]:
class FeatureAdapter:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)
        self.tiles_hist_short = deque(maxlen=cfg.h_short)
        self.tiles_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)
        self.tiles_freq_short = defaultdict(int)
        self.tiles_freq_long = defaultdict(int)

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tiles_hist_short,
            self.tiles_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tiles_freq_short,
            self.tiles_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

    def update_history(self, vid: int, tiles: list[int]):       
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = tuple(tiles) if tiles is not None else None

        if tiles is None:
            return

        self._update_window(self.tiles_hist_short, self.tiles_freq_short, tiles)
        self._update_window(self.tiles_hist_long, self.tiles_freq_long, tiles)

        for tile in tiles:
            self._update_window(self.tile_hist_short, self.tile_freq_short, (vid, tile))
            self._update_window(self.tile_hist_long, self.tile_freq_long, (vid, tile))

    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1

In [41]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, feature_adapter: FeatureAdapter, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.capacity = int(
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.n_tiles +
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.tiles_per_viewport            
        )
        self.n_features = self.capacity + 1

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        print(f"NetworkAdapter initialized with capacity: {self.capacity}")

        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]

    def _rank_cached_videos(self, bitmap: np.ndarray) -> list[int]:
        cached_mask = np.any(bitmap[:, 0, :, :], axis=(1, 2))
        cached_vids = np.where(cached_mask)[0].tolist()
        ranked = sorted(
            cached_vids,
            key=lambda vid: self.features.video_freq_long.get(vid, 0),
            reverse=True
        )

        pad = [-1] * (self.capacity - len(ranked))
        return ranked + pad

    def _cached_viewport_tiles(self, bitmap: np.ndarray, vid: int, k: int) -> list[int]:
        if vid == -1:
            return [-1] * k

        tile_counts = bitmap[vid, 1, :, :].sum(axis=1)
        cached_tiles = np.flatnonzero(tile_counts > 0).astype(int).tolist()

        # Rank by count desc, then tile id asc for determinism
        ranked = sorted(
            cached_tiles,
            key=lambda t: (-int(tile_counts[t]), t)
        )

        # Pad if fewer than k tiles
        ranked.extend([-1] * max(0, k - len(ranked)))

        # print(f"Tile counts for video {vid}: {tile_counts}")
        # print(f"Cached tiles for video {vid}: {ranked[:k]}")

        return ranked[:k]

    def _decode_action(self, action_idx: int) -> tuple[int, int]:
        """
        Split flat index into (a1, a2):
          a1 in [0, C]  : base-layer decision (A1)
          a2 in [0, C*k): enh-tile decision (A2)
        """
        a2_size = self.C * self.k + 1
        return action_idx // a2_size, action_idx % a2_size

    def get_video_cache_idx(self, vid: int) -> int:
        for idx, v in enumerate(self.video_cache_index):
            if v == vid:
                return idx
        return -1

    def build_observation(self, request: Dict) -> np.ndarray:
        """
        Builds the state vector as in the paper:
          [ x_s (C), y_s (C*k), z_s (1), x_l (C), y_l (C*k), z_l (1) ]
        where:
          - x_s/x_l: counts of requests for cached base videos (short/long windows)
          - y_s/y_l: counts of requests for cached enh tiles per video (short/long)
          - z_s/z_l: counts for the currently examined item (video or tile)
        Total dim = 10*C + 2 when k=4.
        """
        C, k = self.C, self.k

        # Initialize feature blocks
        x_s = np.zeros(C, dtype=np.float32)
        x_l = np.zeros(C, dtype=np.float32)

        y_s = np.zeros(C * k, dtype=np.float32)
        y_l = np.zeros(C * k, dtype=np.float32)

        #Takes in consideration the ranked cached videos using the long-term popularity video
        cache_bitmap = self.env.mec_cache.get_cache_bitmap()
        ranked_cached = self._rank_cached_videos(cache_bitmap)[:C]

        # print("Ranked cached videos:", ranked_cached)

        for i, vid in enumerate(ranked_cached):
            if vid == -1:
                continue
            x_s[i] = self.features.video_freq_short.get(vid, 0)
            x_l[i] = self.features.video_freq_long.get(vid, 0)

            tiles = self._cached_viewport_tiles(cache_bitmap, vid, k)

            base_off = i * k
            for j, t in enumerate(tiles):
                if t == -1:
                    continue
                y_s[base_off + j] = self.features.tile_freq_short.get((vid, t), 0)
                y_l[base_off + j] = self.features.tile_freq_long.get((vid, t), 0)

            # print(f"Video {vid} cached tiles for y_s/y_l:", tiles)

        video = request["video"]
        viewport = request["viewport"] if request["viewport"] is not None else []

        z_s = np.array([
            self.features.video_freq_short.get(video, 0) if request.get("layer", 0) == 0
            else sum(self.features.tile_freq_short.get((video, t), 0) for t in viewport)
        ], dtype=np.float32)
        z_l = np.array([
            self.features.video_freq_long.get(video, 0) if request.get("layer", 0) == 0
            else sum(self.features.tile_freq_long.get((video, t), 0) for t in viewport)
        ], dtype=np.float32)

        return np.concatenate([x_s, x_l, y_s, y_l, z_s, z_l], axis=0)

    def evict_video(self, bitmap: np.ndarray, v: int):
        for layer, tile_id, gop_id in np.argwhere(bitmap[v] == 1):
            bitmap[v, layer, tile_id, gop_id] = 0

    def apply_action(self, action_idx: int, request: Dict, bitmap: np.ndarray) -> Dict:
        """
        Implements the paper's action space:
          - A1 (size C+1): when base is not cached. a0 = no-op; a_i evicts the i-th cached video and caches the requested one.
          - A2 (size k+1): when base is cached but viewport differs. a0 = no-op; a_j replaces the j-th cached enh tile with the j-th requested tile.
        """
        vid, gop = request["video"], request["gop"]
        viewport = request["viewport"] if request["viewport"] is not None else []

        print(
            f"Applying action: {action_idx}\n"
            f"Ranked videos: {self.video_cache_index}"
        )
        # ranked_videos = self._rank_cached_videos(bitmap)[:self.C]

        if action_idx == 0:
            return  # No-op

        if action_idx <= self.C:
            victim_vid = self.video_cache_index[action_idx - 1]

            if victim_vid != -1:
                self.evict_video(bitmap, victim_vid)
            self.env.mec_cache.cache_new_video(vid, gop_id=0, layer=0)

            return

        if viewport is None:
            return  # No viewport to process

        vid_idx = self.get_video_cache_idx(vid)
        
        tile_block_start = self.C + vid_idx * 4
        tile_idx = action_idx - (tile_block_start + 1) # 0..3

        k = self.tile_cache_index[vid_idx][tile_idx]

        if k == -1:
            k = self.tile_cache_index[vid_idx][tile_idx] = viewport[tile_idx]
        else:            
            self.evict_tile(bitmap, vid, k)

        self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_id=k)

    def evict_tile(self, bitmap: np.ndarray, vid: int, tile_id: int):
        for gop_id in np.where(bitmap[vid, 1, tile_id, :] == 1)[0]:
            bitmap[vid, 1, tile_id, gop_id] = 0

    def _apply_tile_action(
        self, 
        action_idx: int, 
        vid: int, 
        gop: int, 
        viewport_tiles: list[int], 
        bitmap: np.ndarray
    ):
        """
        Apply the tile-layer action (A2) as per the paper's description.
        """
        n_tiles_to_replace = int(action_idx)

        if n_tiles_to_replace == 0:
            return  # No-op
        
        if viewport_tiles is None:
            return  # No viewport to process
        
        tile_freq = defaultdict(int)
        for tiles in self.features.tile_hist_long:
            if tiles is None:
                continue
            for t in tiles:
                tile_freq[int(t)] += 1

        cached_tile_mask = np.any(bitmap[vid, 1, :, :], axis=1)
        cached_tiles = np.flatnonzero(cached_tile_mask).astype(int).tolist()

        if not cached_tiles:
            return  # No cached tiles to replace
        
        # Candidates: viewport tiles not already cached
        viewport_unique = list(dict.fromkeys(viewport_tiles))
        viewport_candidates = [t for t in viewport_unique if t not in cached_tiles]

        # Sort cached tiles by ascending frequency (least frequent first)
        cached_sorted = sorted(
            cached_tiles,
            key=lambda t: (tile_freq.get(t, 0), t)
        )

        # Sort viewport candidates by descending frequency (most frequent first)
        viewport_sorted = sorted(
            viewport_candidates,
            key=lambda t: (tile_freq.get(t, 0), t),
            reverse=True
        )

        n = min(n_tiles_to_replace, len(cached_sorted), len(viewport_sorted))
        if n <= 0:
            return

        # Evict n least frequent cached tiles
        def _evict_tile(tile_id: int):
            for gop_id in np.where(bitmap[vid, 1, tile_id, :] == 1)[0]:
                key = (vid, 1, int(tile_id), int(gop_id))
                self.env.mec_cache.policy.remove(key)
                bitmap[vid, 1, tile_id, gop_id] = 0

        # Cache new tiles from viewport
        def _cache_tile(tile_id: int):
            # Assumes cache_new_video supports layer and tile_id for enh tiles
            self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_id=int(tile_id))

        for i in range(n):
            old_tile = cached_sorted[i]
            new_tile = viewport_sorted[i]
            _evict_tile(old_tile)
            _cache_tile(new_tile)

    def last_sample_replication(
        self, 
        vid: int, 
        gop: int, 
        viewport: list[int], 
        bitmap: np.ndarray
    ):
        for tile in viewport:
            self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_id=tile)

In [42]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()

    # 2. Initialize Environment
    du_caches = []

    unit_mapper = CacheUnitMapper(
        cache_capacity_mb=cfg.cache_capacity_b / 1e6,
        num_gops=cfg.n_gops,
        num_tiles=cfg.n_tiles,
        viewport_tiles=4,  # assuming viewport with 4 tiles
        base_tile_mb=2e6 / 1e6 / cfg.n_tiles,
        enh_tile_mb=15e6 / 1e6 / cfg.n_tiles
    )

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_b,
        # policy=SvcLruPolicy(max_size=max_capacity)
        unit_mapper=unit_mapper
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.zipf_alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    # 3. Initialize Agent
    agent = DQNAgent(cfg)

    obs, info = env.reset()
    feature_adapter = FeatureAdapter(env, cfg)
    net_adapter = NetworkAdapter(env, feature_adapter, cfg)

    for episode in range(cfg.n_episodes):

        obs, info = env.reset()
        feature_adapter.reset_history()
        
        cache_hits = 0
        cache_misses = 0
        total_reward = 0.0
        avg_psnr = []

        for step in count():

            # Track per-user state/action/reward for training
            state_by_user = {}
            action_by_user = {}
            reward_by_user = {}

            # Get active users (not finished all GOPs)
            reqs_state = info['users_requests']
            active_users = [
                req for req in reqs_state if req['gop'] < cfg.n_gops
            ]

            # Count active users per DU
            active_users_per_du = Counter(req["p"] for req in active_users)

            # Get current cache bitmap ( Videos x Layers x Tiles x GOPs )
            bitmap = net_adapter.env.mec_cache.get_cache_bitmap()

            # Main Loop. Process each active user request
            for req in active_users:
                u, p, v, g, tiles = req['u'], req['p'], req['video'], req['gop'], req["tiles"]
                viewport = req['viewport'] if req['viewport'] is not None else []

                # Check if any requested tile is missing in cache
                missing_base_layer = np.all(bitmap[v, 0, :, :] == 0)

                # print(f"Step {step} User {u} Video {v} GOP {g} Missing Base-layer: {missing_base_layer}")

                if g == 0 and missing_base_layer:
                    # Build current state
                    state = net_adapter.build_observation(req)

                    # Select action and apply it
                    action_idx, q_actions = agent.select_action(state, g=g)
                    net_adapter.apply_action(action_idx, req, bitmap)

                    # Compute reward for the step
                    step_reward = (
                        cfg.r_base * (
                            1 if np.any(bitmap[v, 0, :, g] == 1) else 0
                        ) +
                        cfg.r_enh * sum(
                            1 for tile in tiles if tile['layer'] == 1 and 
                            bitmap[v, 1, tile['tile'], g] == 1
                        )
                    ) / len(tiles) if len(tiles) > 0 else 0.0

                    state_by_user[u] = state
                    action_by_user[u] = action_idx
                    reward_by_user[u] = step_reward

                    total_reward += step_reward

                elif not missing_base_layer:
                    idx = net_adapter.get_video_cache_idx(v)

                    # Build current state
                    state = net_adapter.build_observation(req)

                    # Select action and apply it
                    action_idx, q_actions = agent.select_action(state, idx=idx, g=g)
                    net_adapter.apply_action(action_idx, req, bitmap)

                    # Compute reward for the step
                    step_reward = (
                        cfg.r_base * sum(
                            1 for tile in tiles if tile['layer'] == 0 and 
                            bitmap[v, 0, tile['tile'], g] == 1
                        ) +
                        cfg.r_enh * sum(
                            1 for tile in tiles if tile['layer'] == 1 and 
                            bitmap[v, 1, tile['tile'], g] == 1
                        )
                    ) / len(tiles) if len(tiles) > 0 else 0.0

                    state_by_user[u] = state
                    action_by_user[u] = action_idx
                    reward_by_user[u] = step_reward

                    total_reward += step_reward

                # Compute Cache Performance
                ch = np.sum(
                    [bitmap[v, tile["layer"], tile["tile"], g] for tile in tiles],
                    dtype=np.int64,
                )

                cache_hits += ch
                cache_misses += len(tiles) - ch
                # hit_ratio = (ch / len(tiles)) if len(tiles) > 0 else 0.0

                print(
                    f"Step {step}, User {u}, "
                    f"Video: {v}, Gop: {g}, Viewport: {viewport} \n"
                    f"Cache Hits: {cache_hits}, "
                    f"Misses: {cache_misses}, "
                    # f"Hit Ratio: {hit_ratio:.2f}\n"
                )

                # Update feature history for the requested video and viewport
                feature_adapter.update_history(v, viewport)

            # Last-sample replication for the next requested tiles in viewport
            for req in active_users:
                v, g, viewport = req['video'], req['gop'], req['viewport']

                if np.any(bitmap[v, 0, :, :] == 0):
                    continue  # base layer not cached

                if g >= cfg.n_gops - 1 or viewport is None:
                    continue # no more GOPs or viewport

                net_adapter.last_sample_replication(v, g+1, viewport, bitmap)

            # Updates in the state after processing all active users
            reqs_next_state = net_adapter.env.users_env.step(
                None, bitmap
            )
            reqs_next_by_user = {r["u"]: r for r in reqs_next_state}

            for req in active_users:
                u = req['u']

                if u not in reqs_next_by_user:
                    continue

                if u not in state_by_user or u not in action_by_user:
                    continue

                req_next = reqs_next_by_user[u]
                next_state = net_adapter.build_observation(req_next)

                # --- TRAINING / HISTORY UPDATE ---
                if u in state_by_user and u in action_by_user:
                    agent.remember(
                        state_by_user[u], 
                        action_by_user[u], 
                        reward_by_user[u], 
                        next_state, 
                        done=net_adapter.env.users_env.user_is_done(u)
                    )

            if step % agent.nb_interval == 0:
                agent.train_step()
                agent.update_target()

            done = (net_adapter.env.users_env.users_done >= net_adapter.env.users_env.n_users)
            if done:
                break

            info = {
                'users_requests': reqs_next_state
            }

            print(f"Step {step}, Active Users: {len(active_users)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Next Request State: {reqs_next_state}")
            print("----------------------------------------------------------------")

        agent.update_epsilon()
        
        filename = (
            f"drl_cache_predictor_E{cfg.n_episodes}_U{cfg.n_users}_"
            f"V{cfg.n_videos}_G{cfg.n_gops}_L{cfg.n_layers}_n{cfg.n}_m{cfg.m}_"
            f"cap{cfg.cache_size}_AR{cfg.arrival_rate}_Z{cfg.zipf_alpha}.csv"
        )

        save_training_results(
            # path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            path_=r'c:\Users\es25591\Workspace\CacheVideoPredict360\Results',
            filename=filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            agent=agent
        )

--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 24000
Step 0, Active Users: 0
----------------------------------------------------------------
Step 1, Active Users: 0
----------------------------------------------------------------
Step 2, Active Users: 0
----------------------------------------------------------------
Step 3, Active Users: 0
----------------------------------------------------------------
Step 4, Active Users: 0
----------------------------------------------------------------
Step 5, Active Users: 0
----------------------------------------------------------------
Step 6, Active Users: 0
----------------------------------------------------------------
Step 7, Active Users: 0
----------------------------------------------------------------
Step 8, Active Users: 0
----------------------------------------------------------------
Applying action: 44
Ranked videos: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,

KeyboardInterrupt: 

In [ ]:
import itertools
import numpy as np

def build_action_space(C: int, k: int):
    # A1: base-not-cached actions (0 = no-op, 1..C evict ranked i-th video)
    A1 = list(range(C + 1))
    
    # A2: enhance-tile actions (C*k entries, each replaces one tile position)
    A2 = list(range(C * k))
    
    # Cartesian product (indices)
    A = list(itertools.product(A1, A2))
    
    return A1, A2, A  # A has size (C+1) * (C*k) = 5C+1 when k=4

# Example usage
C, k = 2, 4  # cache can hold 2 videos; viewport budget k=4
A1, A2, A = build_action_space(C, k)
print("A1 (C+1):", A1)
print("A2 (C*k):", A2)
print("Total |A|:", len(A), "== (C+1)*C*k =", (C+1)*C*k)

# One-hot encoding helper for the flat index in [0, 5C]
def one_hot_action(idx: int, C: int, k: int):
    size = (C + 1) + C * k  # 5C+1 when k=4
    vec = np.zeros(size, dtype=np.float32)
    vec[idx] = 1.0
    
    return vec

# Example: pick A1 action i=1 (evict first video) and A2 action m=3 (replace tile slot 3)
a1_choice, a2_choice = 1, 3
flat_idx = a1_choice if a1_choice <= C else (C + 1 + a2_choice)
action_vec = one_hot_action(flat_idx, C, k)

print("Flat index:", flat_idx)
print("One-hot vector:", action_vec)

A1 (C+1): [0, 1, 2]
A2 (C*k): [0, 1, 2, 3, 4, 5, 6, 7]
Total |A|: 24 == (C+1)*C*k = 24
Flat index: 1
One-hot vector: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
